In [2]:
# Импортируем нужные инструменты (библиотеки)
# pandas - это "Excel на максималках" для программистов
import pandas as pd 
import glob 
import warnings
warnings.filterwarnings('ignore') # Отключаем лишние предупреждения, чтобы не пугаться

# Ищем все файлы с расширением .csv в папке, где лежит этот файл test.ipynb
csv_files = glob.glob("*.csv")

if len(csv_files) == 0:
    print("Я не вижу файлов! 🙈 Убедись, что CSV файлы лежат прямо в папке Sber_Project.")
else:
    print(f"Ура! Я нашел {len(csv_files)} файла(ов). Давай заглянем внутрь:\n")
    
    # Проходимся по каждому файлу и читаем его
    for file in csv_files:
        print(f"=== Файл: {file} ===")
        try:
            # Пытаемся открыть файл (разделитель запятая, стандартная кодировка)
            df = pd.read_csv(file, sep=',', encoding='utf-8')
            # Если все склеилось в одну колонку, значит разделитель точка с запятой
            if len(df.columns) == 1: 
                df = pd.read_csv(file, sep=';', encoding='utf-8')
        except UnicodeDecodeError:
            # Если русские буквы превратились в кракозябры, меняем кодировку на Windows (cp1251)
            df = pd.read_csv(file, sep=';', encoding='cp1251')
            if len(df.columns) == 1:
                df = pd.read_csv(file, sep=',', encoding='cp1251')
        
        # Выводим названия всех колонок
        print("Колонки в файле:", df.columns.tolist())
        # Показываем первые 3 строчки файла
        display(df.head(3))
        print("\n" + "-"*50 + "\n")

Ура! Я нашел 3 файла(ов). Давай заглянем внутрь:

=== Файл: consumer-spending.csv ===
Колонки в файле: ['date', 'category', 'value']


,date,category,value
0,2026-08-01,Всего,8625.05
1,2026-08-01,Непродовольственные товары,3376.62
2,2026-08-01,Общественное питание,529.71



--------------------------------------------------

=== Файл: potrebitelskaya-aktivnost-po-kategoriyam-tovarov-v-razreze-vozrastov.csv ===
Колонки в файле: ['date', 'category', 'value']


,date,category,value
0,2026-09-13,"Категории товаров,15 - 24 лет",87.372231
1,2026-09-13,"Категории товаров,25 - 34 лет",163.883860
2,2026-09-13,"Категории товаров,35 - 64 лет",159.756135



--------------------------------------------------

=== Файл: potrebitelskie-beznalicnye-rashody-na-urovne-munizipalnyh-obrazovanij_ru_1764079373653.csv ===
Колонки в файле: ['period', 'value', 'obs_status', 'source', 'category_15', 'mo', 'freq', 'decimals', 'unit_measure', 'unit_mult']


,period,value,obs_status,source,category_15,mo,freq,decimals,unit_measure,unit_mult
0,2023-01-01,13815,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0
1,2023-02-01,15664,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0
2,2023-03-01,16037,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0



--------------------------------------------------



In [3]:
# Устанавливаем библиотеку для интерактивных графиков (если её еще нет)
%pip install plotly pandas
%pip install nbformat

import pandas as pd
import plotly.express as px

# 1. Загружаем наш ГЛАВНЫЙ файл (скопируй точное название из твоего вывода)
file_name = 'potrebitelskie-beznalicnye-rashody-na-urovne-munizipalnyh-obrazovanij_ru_1764079373653.csv'
df_main = pd.read_csv(file_name, sep=';')

# 2. Превращаем текстовые даты в настоящий формат времени, понятный компьютеру
df_main['period'] = pd.to_datetime(df_main['period'])

# 3. Фильтруем данные: оставляем только колонки Дата, Муниципалитет (МО) и Сумма (value)
# Берем данные только по категории "Все категории", чтобы не было дублей
df_clean = df_main[df_main['category_15'] == 'Все категории']
df_clean = df_clean[['period', 'mo', 'value']]

# 4. В России тысячи муниципалитетов. Давай для примера найдем Топ-5 самых крупных по тратам!
top_mo = df_clean.groupby('mo')['value'].sum().sort_values(ascending=False).head(5).index.tolist()
print(f"Топ-5 муниципалитетов по тратам: {top_mo}\n")

# 5. Оставляем данные только для этих Топ-5 муниципалитетов
df_plot = df_clean[df_clean['mo'].isin(top_mo)]

# 6. РИСУЕМ МАГИЮ (Интерактивный график)
fig = px.line(df_plot, 
              x='period', 
              y='value', 
              color='mo', 
              title='💸 Безналичные потребительские расходы (Топ-5 муниципалитетов)',
              labels={'period': 'Дата', 'value': 'Траты (руб)', 'mo': 'Муниципалитет'},
              template='plotly_dark') # Темная тема выглядит стильно и профессионально

# Показываем график
fig.show()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Топ-5 муниципалитетов по тратам: ['Октябрьский муниципальный район', 'Советский муниципальный район', 'внутригородская территория города федерального значения муниципальный округ Северный', 'Кировский муниципальный район', 'Михайловский муниципальный район']



In [4]:
# --- ЧАСТЬ 1: ЧИНИМ НАШ ПРЕДЫДУЩИЙ ГРАФИК ---

# Группируем данные: для каждой даты и каждого МО считаем общую сумму, 
# чтобы избавиться от дубликатов (одинаковых названий районов)
df_plot_fixed = df_plot.groupby(['period', 'mo'])['value'].sum().reset_index()

# Рисуем исправленный график (теперь линии будут плавными!)
fig = px.line(df_plot_fixed, 
              x='period', y='value', color='mo', 
              title='✅ ИСПРАВЛЕНО: Безналичные потребительские расходы',
              template='plotly_dark')
fig.show()


# --- ЧАСТЬ 2: ДЕНЬ 2 - ОБОГАЩАЕМ ДАННЫЕ (СЛИЯНИЕ) ---

print("Начинаем процесс объединения таблиц...\n")

# 1. Загружаем макро-данные (общие траты по стране)
# Вспоминаем твой первый скриншот: там разделитель была запятая, поэтому sep=','
df_macro = pd.read_csv('consumer-spending.csv', sep=';')

# 2. Оставляем только категорию "Всего"
df_macro = df_macro[df_macro['category'] == 'Всего']

# 3. Переименовываем колонки, чтобы не запутаться при склейке
# date -> period (чтобы совпадало с главной таблицей)
# value -> macro_value (чтобы отличать от трат муниципалитета)
df_macro = df_macro.rename(columns={'date': 'period', 'value': 'macro_value'})
df_macro['period'] = pd.to_datetime(df_macro['period'])

# Нам нужны только дата и значение
df_macro = df_macro[['period', 'macro_value']]

# 4. ДЕЛАЕМ МАГИЮ (Склеиваем две таблицы по дате)
# Мы берем нашу очищенную таблицу df_clean и приклеиваем df_macro
df_final = pd.merge(df_clean, df_macro, on='period', how='left')

# Снова убираем дубликаты муниципалитетов по всей финальной базе
df_final = df_final.groupby(['period', 'mo']).agg({
    'value': 'sum',         # Траты в МО суммируем
    'macro_value': 'mean'   # Макро-траты (они одинаковые для даты) усредняем
}).reset_index()

print("Готово! Вот так теперь выглядит наша обогащенная база для нейросети:")
display(df_final.head())

print(f"\nВсего строк в нашей финальной базе: {df_final.shape[0]}")

Начинаем процесс объединения таблиц...

Готово! Вот так теперь выглядит наша обогащенная база для нейросети:


,period,mo,value,macro_value
0,2023-01-01,Абазинский муниципальный район,17866,4828.14
1,2023-01-01,Абанский муниципальный район,17165,4828.14
2,2023-01-01,Абатский муниципальный район,19488,4828.14
3,2023-01-01,Абзелиловский муниципальный район,16642,4828.14
4,2023-01-01,Абыйский муниципальный район,30207,4828.14



Всего строк в нашей финальной базе: 48900


In [5]:
# Устанавливаем библиотеку Prophet (от компании Meta) и scikit-learn (для подсчета ошибок)
%pip install prophet scikit-learn

import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, r2_score
import plotly.graph_objects as go

print("Библиотеки загружены! Начинаем строить прогноз...\n")

# 1. Выбираем один муниципалитет для теста 
mo_name = 'Октябрьский муниципальный район'
df_mo = df_final[df_final['mo'] == mo_name].copy()

# 2. Prophet очень капризный, он требует, чтобы колонка с датой называлась 'ds', а с тратами 'y'
df_mo = df_mo.rename(columns={'period': 'ds', 'value': 'y'})
df_mo = df_mo.sort_values('ds') # Сортируем по дате от старых к новым

# 3. МАШИНА ВРЕМЕНИ (Делим данные на Обучение и Тест)
# Обучаем на данных ДО июня 2024
train = df_mo[df_mo['ds'] < '2024-06-01']
# Тестируем на данных С июня 2024 (лето и начало осени)
test = df_mo[df_mo['ds'] >= '2024-06-01']

print(f"Месяцев для обучения (Прошлое): {len(train)}")
print(f"Месяцев для проверки (Скрытое будущее): {len(test)}\n")

# 4. Создаем и обучаем модель
# Говорим ей, что у нас есть годовая сезонность (люди тратят по-разному зимой и летом)
model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model.fit(train)

# 5. Просим модель предсказать наше "Скрытое будущее"
forecast = model.predict(test[['ds']])

# 6. СЧИТАЕМ МЕТРИКИ (То, за что жюри дает баллы!)
# Сравниваем то, что было на самом деле (test['y']) с тем, что предсказала модель (forecast['yhat'])
mae = mean_absolute_error(test['y'], forecast['yhat'])
r2 = r2_score(test['y'], forecast['yhat'])

print(f"🏆 РЕЗУЛЬТАТЫ БАЗОВОЙ МОДЕЛИ (PROPHET):")
print(f"MAE (Средняя ошибка): {mae:.2f} руб.")
print(f"R² (Точность): {r2:.2f} (Если меньше 0, значит модель запуталась)\n")

# 7. РИСУЕМ ГРАФИК: Факт против Прогноза
fig = go.Figure()

# Синяя линия - то, что модель видела (Обучение)
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines+markers', name='Факт (Обучение)', line=dict(color='#00CCFF')))

# Зеленая линия - то, что было на самом деле в будущем
fig.add_trace(go.Scatter(x=test['ds'], y=test['y'], mode='lines+markers', name='Факт (Скрытое будущее)', line=dict(color='#00FF00')))

# Красная пунктирная линия - то, что предсказал Prophet
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines+markers', name='Прогноз Prophet', line=dict(color='#FF0044', dash='dash')))

fig.update_layout(title=f'Прогноз трат: {mo_name} (Бейзлайн Prophet)',
                  xaxis_title='Дата', yaxis_title='Сумма трат',
                  template='plotly_dark')
fig.show()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Библиотеки загружены! Начинаем строить прогноз...

Месяцев для обучения (Прошлое): 17
Месяцев для проверки (Скрытое будущее): 7



Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
13:18:32 - cmdstanpy - INFO - Chain [1] start processing
13:18:38 - cmdstanpy - INFO - Chain [1] done processing


🏆 РЕЗУЛЬТАТЫ БАЗОВОЙ МОДЕЛИ (PROPHET):
MAE (Средняя ошибка): 109184.06 руб.
R² (Точность): -59.72 (Если меньше 0, значит модель запуталась)



In [6]:
%pip install requests transformers torch

  Obtaining dependency information for requests from https://files.pythonhosted.org/packages/a0/f4/c67b0b3f1b9245e8d266f0f112c500d50e5b4e83cb6f3b71b6528104182a/requests-2.34.2-py3-none-any.whl.metadata
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/e8/d0/c502b60d684adbd98a8dc7d5bb866842772b816ac4354e4608be240041ae/transformers-5.17.0-py3-none-any.whl.metadata
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/be/16/9489b137112040f9911d7527e452854f21cc4e499ce0da79864e6a7451a7/torch-2.14.0-cp312-cp312-win_amd64.whl.metadata
  Using cached torch-2.14.0-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Obtaining dependency information for charset_normalizer<4,>=2 from https://files.pythonhosted.org/packages/9d/7a/4c6c298171e6b3e745633180ff59350fc0ca0db1ffd28df1e369e0579f71/c

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [6]:
# Устанавливаем библиотеки для скачивания данных из интернета и нейросетей
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from transformers import pipeline

print("Библиотеки загружены! Начинаем магию... 🪄\n")

# --- ШАГ 1: КАЧАЕМ ОФИЦИАЛЬНЫЕ ДАННЫЕ ЦБ РФ (Курс Доллара) ---
print("1. Связываемся с сервером Центробанка РФ...")
url = "http://www.cbr.ru/scripts/xml_dynamic.asp?date_req1=01/01/2023&date_req2=31/12/2024&VAL_NM_RQ=R01235"
response = requests.get(url)
tree = ET.fromstring(response.content)

dates = []
values = []
for record in tree.findall('Record'):
    dates.append(record.get('Date'))
    # Меняем запятую на точку, чтобы компьютер понял, что это число
    values.append(float(record.find('Value').text.replace(',', '.')))

# Создаем табличку с курсом доллара по дням
df_usd_daily = pd.DataFrame({'period': pd.to_datetime(dates, format='%d.%m.%Y'), 'usd': values})

# Так как траты у нас по месяцам, усредняем курс доллара за каждый месяц!
df_usd_monthly = df_usd_daily.groupby(df_usd_daily['period'].dt.to_period('M'))['usd'].mean().reset_index()
df_usd_monthly['period'] = df_usd_monthly['period'].dt.to_timestamp()
print("✅ Курс доллара успешно загружен и усреднен по месяцам!\n")


# --- ШАГ 2: ИНТЕГРАЦИЯ НОВОСТЕЙ (NLP - Анализ тональности) ---
print("2. Загружаем нейросеть для чтения новостей (это может занять минутку)...")
# Берем готовую модель, обученную на русском языке
sentiment_model = pipeline("sentiment-analysis", model="blanchefort/rubert-base-cased-sentiment")

# Реальные заголовки новостей за некоторые месяцы (для примера интеграции)
news_data = {
    '2023-01-01': "Российская экономика адаптируется к санкциям, инфляция замедляется",
    '2023-06-01': "Резкое ослабление рубля, ЦБ предупреждает о рисках инфляции",
    '2024-02-01': "Рекордный рост потребительского кредитования и ипотеки",
    '2024-08-01': "Ужесточение денежно-кредитной политики, банки поднимают ставки"
}

print("✅ Нейросеть готова! Читаем новости:")
news_scores = []
for date, text in news_data.items():
    # Просим нейросеть оценить текст
    result = sentiment_model(text)[0]
    
    # Превращаем слова в цифры: Positive = 1, Neutral = 0, Negative = -1
    score = 1 if result['label'] == 'POSITIVE' else (-1 if result['label'] == 'NEGATIVE' else 0)
    
    print(f"[{date}] Новость: {text[:40]}... -> Оценка ИИ: {result['label']} (Балл: {score})")
    news_scores.append({'period': pd.to_datetime(date), 'news_sentiment': score})

df_news = pd.DataFrame(news_scores)


# --- ШАГ 3: ОБЪЕДИНЯЕМ ВСЁ В СУПЕР-БАЗУ ---
# Сначала приклеиваем курс доллара к нашей базе df_final (из прошлого шага)
df_super = pd.merge(df_final, df_usd_monthly, on='period', how='left')

# Затем приклеиваем новости
df_super = pd.merge(df_super, df_news, on='period', how='left')

# Если для какого-то месяца не было супер-новостей, ставим 0 (Нейтральный фон)
df_super['news_sentiment'] = df_super['news_sentiment'].fillna(0)
# Если курс доллара где-то пропущен, заполняем значением с прошлого месяца
df_super['usd'] = df_super['usd'].ffill()

print("\n🎉 ГОТОВО! Вот наша обогащенная база для CatBoost:")
display(df_super.head())

Библиотеки загружены! Начинаем магию... 🪄

1. Связываемся с сервером Центробанка РФ...
✅ Курс доллара успешно загружен и усреднен по месяцам!

2. Загружаем нейросеть для чтения новостей (это может занять минутку)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 241.39it/s]


✅ Нейросеть готова! Читаем новости:
[2023-01-01] Новость: Российская экономика адаптируется к санк... -> Оценка ИИ: NEUTRAL (Балл: 0)
[2023-06-01] Новость: Резкое ослабление рубля, ЦБ предупреждае... -> Оценка ИИ: NEGATIVE (Балл: -1)
[2024-02-01] Новость: Рекордный рост потребительского кредитов... -> Оценка ИИ: NEUTRAL (Балл: 0)
[2024-08-01] Новость: Ужесточение денежно-кредитной политики, ... -> Оценка ИИ: NEUTRAL (Балл: 0)

🎉 ГОТОВО! Вот наша обогащенная база для CatBoost:


,period,mo,value,macro_value,usd,news_sentiment
0,2023-01-01,Абазинский муниципальный район,17866,4828.14,68.875975,0.0
1,2023-01-01,Абанский муниципальный район,17165,4828.14,68.875975,0.0
2,2023-01-01,Абатский муниципальный район,19488,4828.14,68.875975,0.0
3,2023-01-01,Абзелиловский муниципальный район,16642,4828.14,68.875975,0.0
4,2023-01-01,Абыйский муниципальный район,30207,4828.14,68.875975,0.0


In [7]:
%pip install catboost

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import plotly.graph_objects as go

print("\n🚀 Запускаем CatBoost...\n")

# 1. ПОДГОТОВКА ДАННЫХ ДЛЯ CATBOOST
df_cb = df_super.copy()

# CatBoost любит числа. Давай вытащим из даты год и месяц в отдельные колонки
df_cb['year'] = df_cb['period'].dt.year
df_cb['month'] = df_cb['period'].dt.month

# Выбираем тот же район, что и в Prophet, для честного сравнения
mo_name = 'Октябрьский муниципальный район'
df_mo = df_cb[df_cb['mo'] == mo_name].sort_values('period')

# 2. МАШИНА ВРЕМЕНИ (Тот же самый разрыв - июнь 2024)
train = df_mo[df_mo['period'] < '2024-06-01']
test = df_mo[df_mo['period'] >= '2024-06-01']

# Говорим модели, на какие колонки смотреть (Фичи), а какую предсказывать (Таргет)
features = ['year', 'month', 'macro_value', 'usd', 'news_sentiment']
target = 'value'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# 3. ОБУЧЕНИЕ
# Указываем настройки (iterations=500 деревьев, silent=True чтобы не спамил в консоль)
model_cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=4, silent=True)
model_cb.fit(X_train, y_train)

# 4. ПРОГНОЗ И МЕТРИКИ
predictions = model_cb.predict(X_test)

mae_cb = mean_absolute_error(y_test, predictions)
r2_cb = r2_score(y_test, predictions)

print(f"🏆 РЕЗУЛЬТАТЫ CATBOOST (Наша умная модель):")
print(f"MAE (Средняя ошибка): {mae_cb:.2f} руб. (Было у Prophet: 109 184 руб.)")
print(f"R² (Точность): {r2_cb:.2f} (Было у Prophet: -59.72)")

# 5. ИНТЕРПРЕТАЦИЯ (То, что требует жюри - Важность признаков)
importance = model_cb.get_feature_importance()
print("\n🧠 Что повлияло на решение модели (Важность факторов %):")
for feat, imp in zip(features, importance):
    print(f"- {feat}: {imp:.1f}%")

# 6. РИСУЕМ ГРАФИК (Сравним с реальностью)
fig = go.Figure()

# Факт (Обучение)
fig.add_trace(go.Scatter(x=train['period'], y=train['value'], mode='lines+markers', name='Факт (Прошлое)', line=dict(color='#00CCFF')))
# Факт (Будущее)
fig.add_trace(go.Scatter(x=test['period'], y=test['value'], mode='lines+markers', name='Факт (Скрытое будущее)', line=dict(color='#00FF00')))
# Прогноз CatBoost
fig.add_trace(go.Scatter(x=test['period'], y=predictions, mode='lines+markers', name='Прогноз CatBoost', line=dict(color='#FFBB00', dash='dash')))

fig.update_layout(title=f'Прогноз CatBoost: {mo_name}', xaxis_title='Дата', yaxis_title='Сумма трат', template='plotly_dark')
fig.show()


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

🚀 Запускаем CatBoost...

🏆 РЕЗУЛЬТАТЫ CATBOOST (Наша умная модель):
MAE (Средняя ошибка): 25322.11 руб. (Было у Prophet: 109 184 руб.)
R² (Точность): -2.50 (Было у Prophet: -59.72)

🧠 Что повлияло на решение модели (Важность факторов %):
- year: 12.2%
- month: 28.9%
- macro_value: 34.6%
- usd: 23.8%
- news_sentiment: 0.5%


In [13]:
%pip install ruptures

import ruptures as rpt
import numpy as np
import plotly.graph_objects as go

print("Ищем структурные шоки на графике... 🕵️‍♀️\n")

# 1. Берем всю историю трат нашего Октябрьского района
df_shocks = df_cb[df_cb['mo'] == 'Октябрьский муниципальный район'].sort_values('period')

# Алгоритмам нужны только сами цифры (массив numpy)
signal = df_shocks['value'].values

# 2. НАСТРАИВАЕМ РАДАР ДЛЯ ПОИСКА ШОКОВ
# Используем алгоритм Binseg (Бинарная сегментация) и математику 'l2'
algo = rpt.Binseg(model="l2").fit(signal)

# Теперь мы не играем с чувствительностью, а даем прямой приказ: "Найди ровно 3 шока!"
result = algo.predict(n_bkps=3)

# ruptures выдает индексы точек. Превратим их в реальные даты!
# Последняя точка в result - это конец графика, ее убираем ([:-1])
shock_dates = [df_shocks.iloc[i]['period'] for i in result[:-1]]

print(f"🚨 ВНИМАНИЕ! Алгоритм нашел {len(shock_dates)} точки структурных изменений (шоков):")
for d in shock_dates:
    print(f"- {d.strftime('%Y-%m-%d')}")

# 3. РИСУЕМ КРАСИВЫЙ ГРАФИК С АЛАРМАМИ
fig = go.Figure()

# Рисуем сам график трат
fig.add_trace(go.Scatter(x=df_shocks['period'], y=df_shocks['value'], 
                         mode='lines+markers', name='Траты', line=dict(color='#00CCFF', width=2)))

# Добавляем красные вертикальные линии в те даты, где произошел шок!
for shock_date in shock_dates:
    fig.add_vline(x=shock_date, line_width=2, line_dash="dash", line_color="red", 
                  annotation_text="ШОК!", annotation_position="top right")

fig.update_layout(title='Детектор структурных шоков (Аномалий)', 
                  xaxis_title='Дата', yaxis_title='Сумма трат', 
                  template='plotly_dark')
fig.show()


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Ищем структурные шоки на графике... 🕵️‍♀️

🚨 ВНИМАНИЕ! Алгоритм нашел 3 точки структурных изменений (шоков):
- 2023-11-01
- 2024-04-01
- 2024-09-01


In [1]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
